In [3]:
import sys
from pathlib import Path

# Project root directory
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\Projects\patient-readmission


In [6]:
from src.preprocess import (
    replace_question_marks,
    remove_leakage_columns
)

print("Import successful")

Import successful


In [4]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path(
    "../data/diabetic_data.csv"
)

df = pd.read_csv(DATA_PATH)

print(df.shape)

(101766, 50)


In [5]:
from src.preprocess import (
    replace_question_marks
)

df = replace_question_marks(df)

In [7]:
question_mark_total = (
    df.astype(str)
      .eq("?")
      .sum()
      .sum()
)

print(question_mark_total)

0


In [8]:
missing_report = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum(),
})

missing_report["missing_pct"] = (
    missing_report["missing_count"]
    / len(df)
    * 100
).round(2)

missing_report = (
    missing_report
    .sort_values(
        by="missing_pct",
        ascending=False
    )
)

missing_report.head(20)

,column,missing_count,missing_pct
weight,weight,98569,96.86
max_glu_serum,max_glu_serum,96420,94.75
A1Cresult,A1Cresult,84748,83.28
medical_specialty,medical_specialty,49949,49.08
payer_code,payer_code,40256,39.56
race,race,2273,2.23
diag_3,diag_3,1423,1.40
diag_2,diag_2,358,0.35
diag_1,diag_1,21,0.02
patient_nbr,patient_nbr,0,0.00


In [9]:
missing_report.to_csv(
    "../reports/data_quality_report.csv",
    index=False
)

In [10]:
duplicate_rows = (
    df.duplicated().sum()
)

print(
    f"Duplicate rows: "
    f"{duplicate_rows:,}"
)

Duplicate rows: 0


In [11]:
print(
    "Unique Patients:",
    df["patient_nbr"].nunique()
)

print(
    "Unique Encounters:",
    df["encounter_id"].nunique()
)

Unique Patients: 71518
Unique Encounters: 101766


In [12]:
from src.preprocess import (
    remove_leakage_columns
)

df_clean = remove_leakage_columns(df)

print(df_clean.shape)

(101766, 48)


In [13]:
[
    col
    for col in [
        "encounter_id",
        "patient_nbr"
    ]
    if col in df_clean.columns
]

[]

In [14]:
quality_report = pd.DataFrame({
    "column": df_clean.columns,
    "dtype": df_clean.dtypes.astype(str),
    "missing_count": df_clean.isna().sum()
})

quality_report["missing_pct"] = (
    quality_report["missing_count"]
    / len(df_clean)
    * 100
).round(2)

quality_report.sort_values(
    by="missing_pct",
    ascending=False
).head(20)

,column,dtype,missing_count,missing_pct
weight,weight,object,98569,96.86
max_glu_serum,max_glu_serum,object,96420,94.75
A1Cresult,A1Cresult,object,84748,83.28
medical_specialty,medical_specialty,object,49949,49.08
payer_code,payer_code,object,40256,39.56
race,race,object,2273,2.23
diag_3,diag_3,object,1423,1.40
diag_2,diag_2,object,358,0.35
diag_1,diag_1,object,21,0.02
time_in_hospital,time_in_hospital,int64,0,0.00


In [15]:
column_actions = pd.DataFrame({
    "column": quality_report["column"]
})

column_actions["recommended_action"] = "keep"

column_actions.loc[
    column_actions["column"]
    == "weight",
    "recommended_action"
] = "drop"

column_actions.loc[
    column_actions["column"]
    == "payer_code",
    "recommended_action"
] = "drop"

column_actions.loc[
    column_actions["column"]
    == "medical_specialty",
    "recommended_action"
] = "drop"

column_actions.head()

,column,recommended_action
race,race,keep
gender,gender,keep
age,age,keep
weight,weight,drop
admission_type_id,admission_type_id,keep


In [16]:
cleaning_summary = {
    "rows": len(df_clean),
    "columns_after_leakage_removal":
        df_clean.shape[1],
    "duplicate_rows":
        duplicate_rows,
    "question_marks_removed":
        "Yes"
}

cleaning_summary

{'rows': 101766,
 'columns_after_leakage_removal': 48,
 'duplicate_rows': np.int64(0),
 'question_marks_removed': 'Yes'}

In [17]:
pd.DataFrame(
    [cleaning_summary]
).to_csv(
    "../reports/cleaning_summary.csv",
    index=False
)